In [ ]:
"""
Nepali Summarization - Generate Summaries for Specific Test Entries
Adaptive generation length based on reference summaries for better ROUGE evaluation.
"""

import os
import json
import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass
import sentencepiece as spm
from tqdm import tqdm

# ============================================================================
# CONFIGURATION
# ============================================================================

@dataclass
class GenConfig:
    MODEL_PATH: str = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\finetuned-summarization\best_model.pt"
    TOKENIZER_PATH: str = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\new-tokenizers\bpe-16-updated.model"
    TEST_JSONL: str = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\Filtered\test_filtered.jsonl"
    OUTPUT_JSON: str = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\results\generated_summaries.json"
    
    BLOCK_SIZE: int = 1024
    TOP_K: int = 40
    TEMPERATURE: float = 0.7
    
    # List of indices of test entries to generate summaries for
    TARGET_INDICES: list = (0, 5, 10, 100, 500, 600, 700)  # Example: first, sixth, and eleventh entries

PROMPT_TEMPLATE = "यो लेखको संक्षेप गर्नुहोस्:\n{text}\nसारांश:\n"

# ============================================================================
# MODEL ARCHITECTURE
# ============================================================================

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.c_attn = nn.Linear(cfg.n_embd, 3 * cfg.n_embd)
        self.c_proj = nn.Linear(cfg.n_embd, cfg.n_embd)
        self.n_head = cfg.n_head
        self.n_embd = cfg.n_embd

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.c_proj(y.transpose(1, 2).contiguous().view(B, T, C))

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.c_fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd)
        self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd)

    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 16384
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.config = cfg
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(cfg.vocab_size, cfg.n_embd),
            wpe=nn.Embedding(cfg.block_size, cfg.n_embd),
            h=nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)]),
            ln_f=nn.LayerNorm(cfg.n_embd),
        ))
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, input_ids):
        B, T = input_ids.size()
        pos = torch.arange(0, T, dtype=torch.long, device=input_ids.device)
        x = self.transformer.wte(input_ids) + self.transformer.wpe(pos)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        return self.lm_head(x)

# ============================================================================
# SUMMARY GENERATION
# ============================================================================

def generate_summary(model, sp, text, reference_summary, cfg, device, length_margin=0.2):
    """
    Generate summary using top-k sampling.
    Length is adjusted to reference_summary token length ± margin.
    """
    model.eval()
    
    prompt = PROMPT_TEMPLATE.format(text=text)
    tokens = sp.encode(prompt)
    input_ids = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)
    generated = input_ids.clone()
    
    # Determine target length from reference
    ref_len = len(sp.encode(reference_summary)) if reference_summary else cfg.BLOCK_SIZE // 8
    min_len = max(1, int(ref_len * (1 - length_margin)))
    max_len = max(1, int(ref_len * (1 + length_margin)))
    
    with torch.no_grad():
        for _ in range(max_len):
            if generated.size(1) - len(tokens) >= max_len:
                break
            logits = model(generated)
            logits = logits[:, -1, :] / cfg.TEMPERATURE
            top_k_logits, top_k_indices = torch.topk(logits, min(cfg.TOP_K, logits.size(-1)))
            probs = F.softmax(top_k_logits, dim=-1)
            next_token = torch.gather(top_k_indices, -1, torch.multinomial(probs, 1))
            generated = torch.cat([generated, next_token], dim=1)
            
            # Stop if EOS token is generated AND minimum length satisfied
            if next_token.item() == 0 and (generated.size(1) - len(tokens)) >= min_len:
                break
    
    return sp.decode(generated[0, len(tokens):].tolist())

# ============================================================================
# MAIN FUNCTION
# ============================================================================

def generate_for_targets():
    cfg = GenConfig()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Load tokenizer
    sp = spm.SentencePieceProcessor()
    sp.load(cfg.TOKENIZER_PATH)
    
    # Load model
    checkpoint = torch.load(cfg.MODEL_PATH, map_location=device, weights_only=False)
    model = GPT(checkpoint['config'])
    model.load_state_dict(checkpoint['model'])
    model.to(device)
    model.eval()
    
    # Load test data
    test_data = []
    with open(cfg.TEST_JSONL, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                test_data.append(json.loads(line))
    
    results = []
    
    for idx in cfg.TARGET_INDICES:
        if idx >= len(test_data):
            continue
        item = test_data[idx]
        generated = generate_summary(model, sp, item['text'], item.get('summary', ""), cfg, device)
        results.append({
            'sample_id': idx,
            'text': item['text'],
            'reference_summary': item.get('summary', ""),
            'generated_summary': generated
        })
    
    # Save generated summaries
    with open(cfg.OUTPUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    
    print(f"✓ Generated summaries saved to {cfg.OUTPUT_JSON}")
    print(f"✓ Total summaries generated: {len(results)}")

if __name__ == "__main__":
    generate_for_targets()


✓ Generated summaries saved to C:\Users\prash\Documents\AI\Major Project\FineTuning\results\generated_summaries.json
✓ Total summaries generated: 7
